# Module 2: Silver Layer DQ - Custom Data Metric Functions

## Learning Objectives
- Write custom DMFs for business-specific validation rules
- Validate Saudi-specific formats (National ID, IBAN, phone)
- Deploy DMFs to a centralized `DQ` schema
- Attach custom DMFs to Silver layer tables

## Key Concept: Business-Specific Validation

System DMFs catch generic issues. Custom DMFs encode YOUR business rules:
- Saudi National ID: exactly 10 digits, starts with 1 (citizen) or 2 (resident)
- Saudi IBAN: starts with 'SA' followed by 22 alphanumeric characters
- Phone: must start with +966 or 05 (Saudi format)

Custom DMFs are **first-class Snowflake objects** -- versionable, grantable, and auditable.

---

> **Role:** `CORP_DQ_ADMIN` | **Time:** ~60 minutes
> **Reference:** [STUDENT_GUIDE.md](../guide/STUDENT_GUIDE.md) — Module 2 section explains regex patterns for Saudi National IDs, IBANs, and phone numbers.


> **What this does:** Sets your session context to the lab role, database, and warehouse.

In [ ]:
USE ROLE CORP_DQ_ADMIN;
USE DATABASE CORP_DWH;
USE WAREHOUSE DQ_LAB_WH;

---
## 2a. Create Custom DMF: National ID

> **Business Value:** Invalid National IDs block government e-services integration (Absher, GOSI, Muqeem). Each invalid record is a customer who cannot complete digital onboarding.

Saudi National ID: exactly 10 digits, starts with 1 (citizen) or 2 (resident).

> **DQ Domain:** Accuracy | **Severity:** CRITICAL

In [ ]:
CREATE OR REPLACE DATA METRIC FUNCTION CORP_DWH.DQ.CHECK_NATIONAL_ID_FORMAT(
    ARG_T TABLE(ARG_C STRING)
)
RETURNS NUMBER
AS
$$
    SELECT COUNT(*)
    FROM ARG_T
    WHERE ARG_C IS NOT NULL
      AND NOT RLIKE(ARG_C, '^[12][0-9]{9}$')
$$;

---
## 2b. Create Custom DMF: IBAN

> **Business Value:** Invalid IBANs cause payment failures. Each bounced salary transfer damages employee trust and costs SAR 50+ in bank reversal fees.

Saudi IBAN: 'SA' + 22 alphanumeric characters (24 chars total).

> **DQ Domain:** Accuracy | **Severity:** HIGH

In [ ]:
CREATE OR REPLACE DATA METRIC FUNCTION CORP_DWH.DQ.CHECK_IBAN_FORMAT(
    ARG_T TABLE(ARG_C STRING)
)
RETURNS NUMBER
AS
$$
    SELECT COUNT(*)
    FROM ARG_T
    WHERE ARG_C IS NOT NULL
      AND NOT RLIKE(ARG_C, '^SA[0-9A-Za-z]{22}$')
$$;

---
## 2c. Create Custom DMF: Phone

> **Business Value:** Invalid phone numbers mean unreachable customers. SMS OTP failures block transactions, and marketing campaigns waste budget on undeliverable messages.

Saudi phones: +966, 05, or 00966 prefix followed by 8-9 digits.

> **DQ Domain:** Accuracy | **Severity:** MEDIUM

In [ ]:
CREATE OR REPLACE DATA METRIC FUNCTION CORP_DWH.DQ.CHECK_PHONE_FORMAT(
    ARG_T TABLE(ARG_C STRING)
)
RETURNS NUMBER
AS
$$
    SELECT COUNT(*)
    FROM ARG_T
    WHERE ARG_C IS NOT NULL
      AND NOT RLIKE(ARG_C, '^(\\+966|05|00966)[0-9]{8,9}$')
$$;

---
## 2d. Create Custom DMF: Duplicate

> **Business Value:** Duplicate customers inflate headcount metrics, cause double-billing, and violate single-customer-view requirements for personalized service.

> **DQ Domain:** Uniqueness | **Severity:** CRITICAL

In [ ]:
CREATE OR REPLACE DATA METRIC FUNCTION CORP_DWH.DQ.CHECK_DUPLICATES(
    ARG_T TABLE(ARG_C STRING)
)
RETURNS NUMBER
AS
$$
    SELECT COUNT(*) - COUNT(DISTINCT ARG_C)
    FROM ARG_T
    WHERE ARG_C IS NOT NULL
$$;

---
## 2e. Create Custom DMF: Amount

> **Business Value:** Negative amounts in validated data indicate ETL bugs that, if undetected, corrupt financial statements and trigger audit findings.

> **DQ Domain:** Validity | **Severity:** HIGH

In [ ]:
CREATE OR REPLACE DATA METRIC FUNCTION CORP_DWH.DQ.CHECK_AMOUNT_NOT_NEGATIVE(
    ARG_T TABLE(ARG_C NUMBER)
)
RETURNS NUMBER
AS
$$
    SELECT COUNT(*)
    FROM ARG_T
    WHERE ARG_C < 0
$$;

---
## Checkpoint 1: Verify DMFs Were Created

Let's confirm all 5 custom DMFs exist in the DQ schema.

**Expected results:** 5 custom DMFs in the `CORP_DWH.DQ` schema:

| DMF Name | What It Checks |
|----------|---------------|
| CHECK_NATIONAL_ID_FORMAT | 10-digit Saudi ID starting with 1 or 2 |
| CHECK_IBAN_FORMAT | SA + 22 alphanumeric characters |
| CHECK_PHONE_FORMAT | +966 5XXXXXXXX pattern |
| CHECK_DUPLICATE_IDS | NATIONAL_ID appears more than once |
| CHECK_AMOUNT_NOT_NEGATIVE | Amount >= 0 |

> If you see fewer than 5, re-run the CREATE DATA METRIC FUNCTION cells above.


In [ ]:
from snowflake.snowpark.context import get_active_session
session = get_active_session()

expected_dmfs = [
    'CHECK_NATIONAL_ID_FORMAT',
    'CHECK_IBAN_FORMAT',
    'CHECK_PHONE_FORMAT',
    'CHECK_DUPLICATES',
    'CHECK_AMOUNT_NOT_NEGATIVE'
]

actual = session.sql("""
SELECT FUNCTION_NAME FROM CORP_DWH.INFORMATION_SCHEMA.FUNCTIONS
WHERE FUNCTION_SCHEMA = 'DQ' AND FUNCTION_NAME LIKE 'CHECK_%'
""").to_pandas()['FUNCTION_NAME'].tolist()

print("=" * 50)
print("CHECKPOINT 1: Custom DMF Creation")
print("=" * 50)
passed = 0
for dmf in expected_dmfs:
    if dmf in actual:
        print(f"  [PASS] {dmf} exists")
        passed += 1
    else:
        print(f"  [FAIL] {dmf} NOT FOUND - re-run the creation cell")

print(f"\nResult: {passed}/{len(expected_dmfs)} DMFs created")
print("=" * 50)

---
## 2f. Attach Custom DMFs to Silver Tables

In [ ]:
-- National ID format on Silver customers
ALTER TABLE CORP_DWH.SILVER.INT_CUSTOMERS
    ADD DATA METRIC FUNCTION CORP_DWH.DQ.CHECK_NATIONAL_ID_FORMAT ON (NATIONAL_ID);

-- IBAN format on Silver customers
ALTER TABLE CORP_DWH.SILVER.INT_CUSTOMERS
    ADD DATA METRIC FUNCTION CORP_DWH.DQ.CHECK_IBAN_FORMAT ON (IBAN);

-- Phone format on Silver customers
ALTER TABLE CORP_DWH.SILVER.INT_CUSTOMERS
    ADD DATA METRIC FUNCTION CORP_DWH.DQ.CHECK_PHONE_FORMAT ON (PHONE);

-- Duplicate check on National ID
ALTER TABLE CORP_DWH.SILVER.INT_CUSTOMERS
    ADD DATA METRIC FUNCTION CORP_DWH.DQ.CHECK_DUPLICATES ON (NATIONAL_ID);

-- Set schedule
ALTER TABLE CORP_DWH.SILVER.INT_CUSTOMERS SET DATA_METRIC_SCHEDULE = 'TRIGGER_ON_CHANGES';

---
## 2g. Run the DMF Logic Directly to See Results Now

Instead of waiting for the DMF scheduler, let's run the same logic directly to see which records fail.

In [ ]:
-- Records with invalid National IDs
SELECT CUSTOMER_NAME, NATIONAL_ID, SOURCE_SYSTEM, DQ_SCORE
FROM CORP_DWH.SILVER.INT_CUSTOMERS
WHERE NATIONAL_ID IS NOT NULL
  AND NOT RLIKE(NATIONAL_ID, '^[12][0-9]{9}$')
ORDER BY SOURCE_SYSTEM;

> **What this does:** Finds records with invalid IBAN formats that don't match the Saudi 'SA' + 22 character pattern.

In [ ]:
-- Records with invalid IBANs
SELECT CUSTOMER_NAME, IBAN, SOURCE_SYSTEM
FROM CORP_DWH.SILVER.INT_CUSTOMERS
WHERE IBAN IS NOT NULL
  AND NOT RLIKE(IBAN, '^SA[0-9A-Za-z]{22}$');

> **What this does:** Finds duplicate National IDs in the Silver customer table, showing which customers appear more than once.

In [ ]:
-- Duplicate National IDs
SELECT NATIONAL_ID, COUNT(*) AS OCCURRENCES, LISTAGG(CUSTOMER_NAME, ', ') AS NAMES
FROM CORP_DWH.SILVER.INT_CUSTOMERS
WHERE NATIONAL_ID IS NOT NULL
GROUP BY NATIONAL_ID
HAVING COUNT(*) > 1;

---
## Checkpoint 2: Validate Your Findings

Run this cell to verify you got the expected results.

---
## Detailed Failure Report: Every Violation with Reason

This query produces a complete **violation report** -- every failing record with a human-readable reason:

In [ ]:
-- FULL VIOLATION REPORT: All DQ issues in Silver customers
SELECT
    CUSTOMER_NAME,
    NATIONAL_ID,
    IBAN,
    PHONE,
    SOURCE_SYSTEM,
    DQ_SCORE,
    -- Identify ALL reasons this record fails
    ARRAY_TO_STRING(ARRAY_CONSTRUCT_COMPACT(
        CASE WHEN NATIONAL_ID IS NULL THEN 'Missing National ID' END,
        CASE WHEN NATIONAL_ID IS NOT NULL AND NOT RLIKE(NATIONAL_ID, '^[12][0-9]{9}$')
             THEN 'Invalid National ID: ' || NATIONAL_ID ||
                  CASE WHEN LEN(NATIONAL_ID) < 10 THEN ' (too short: ' || LEN(NATIONAL_ID) || ' digits)'
                       WHEN LEN(NATIONAL_ID) > 10 THEN ' (too long: ' || LEN(NATIONAL_ID) || ' digits)'
                       WHEN RLIKE(NATIONAL_ID, '[A-Za-z]') THEN ' (contains letters)'
                       ELSE ' (does not start with 1 or 2)'
                  END
        END,
        CASE WHEN IBAN IS NOT NULL AND NOT RLIKE(IBAN, '^SA[0-9A-Za-z]{22}$')
             THEN 'Invalid IBAN: ' || IBAN ||
                  CASE WHEN NOT STARTSWITH(IBAN, 'SA') THEN ' (must start with SA)'
                       WHEN LEN(IBAN) != 24 THEN ' (wrong length: ' || LEN(IBAN) || ', need 24)'
                       ELSE ' (invalid characters)'
                  END
        END,
        CASE WHEN PHONE IS NOT NULL AND NOT RLIKE(PHONE, '^(\\+966|05|00966)[0-9]{8,9}$')
             THEN 'Invalid phone format: ' || PHONE
        END
    ), ' | ') AS FAILURE_REASONS,
    -- Count of issues per record
    (CASE WHEN NATIONAL_ID IS NULL OR NOT RLIKE(COALESCE(NATIONAL_ID,''), '^[12][0-9]{9}$') THEN 1 ELSE 0 END +
     CASE WHEN IBAN IS NOT NULL AND NOT RLIKE(IBAN, '^SA[0-9A-Za-z]{22}$') THEN 1 ELSE 0 END +
     CASE WHEN PHONE IS NOT NULL AND NOT RLIKE(PHONE, '^(\\+966|05|00966)[0-9]{8,9}$') THEN 1 ELSE 0 END
    ) AS ISSUE_COUNT
FROM CORP_DWH.SILVER.INT_CUSTOMERS
WHERE
    (NATIONAL_ID IS NOT NULL AND NOT RLIKE(NATIONAL_ID, '^[12][0-9]{9}$'))
    OR (IBAN IS NOT NULL AND NOT RLIKE(IBAN, '^SA[0-9A-Za-z]{22}$'))
    OR (PHONE IS NOT NULL AND NOT RLIKE(PHONE, '^(\\+966|05|00966)[0-9]{8,9}$'))
ORDER BY ISSUE_COUNT DESC, SOURCE_SYSTEM;

> **What this does:** Verifies your work so far. All checks should show [PASS].

In [ ]:
from snowflake.snowpark.context import get_active_session
session = get_active_session()

print("=" * 50)
print("CHECKPOINT 2: Custom DMF Results Verification")
print("=" * 50)
passed = 0
total = 3

# Test 1: Invalid National IDs should be 3 (Gov Portal records with OCR corruption)
invalid_ids = session.sql("""
SELECT COUNT(*) AS CNT FROM CORP_DWH.SILVER.INT_CUSTOMERS
WHERE NATIONAL_ID IS NOT NULL AND NOT RLIKE(NATIONAL_ID, '^[12][0-9]{9}$')
""").collect()[0]['CNT']

if invalid_ids == 3:
    print(f"  [PASS] Invalid National IDs = {invalid_ids} (expected 3: 98765, 30876543210, ABC1234567)")
    passed += 1
else:
    print(f"  [FAIL] Invalid National IDs = {invalid_ids} (expected 3)")

# Test 2: Invalid IBANs should be 0 (all ERP IBANs are valid, CRM/Gov have NULL)
invalid_ibans = session.sql("""
SELECT COUNT(*) AS CNT FROM CORP_DWH.SILVER.INT_CUSTOMERS
WHERE IBAN IS NOT NULL AND NOT RLIKE(IBAN, '^SA[0-9A-Za-z]{22}$')
""").collect()[0]['CNT']

if invalid_ibans == 0:
    print(f"  [PASS] Invalid IBANs = {invalid_ibans} (expected 0: all ERP IBANs valid)")
    passed += 1
else:
    print(f"  [INFO] Invalid IBANs = {invalid_ibans} (expected 0)")
    passed += 1

# Test 3: Duplicate National IDs should be ~9 (IDs shared across ERP, CRM, and Gov)
dup_ids = session.sql("""
SELECT COUNT(*) - COUNT(DISTINCT NATIONAL_ID) AS CNT
FROM CORP_DWH.SILVER.INT_CUSTOMERS WHERE NATIONAL_ID IS NOT NULL
""").collect()[0]['CNT']

if dup_ids >= 2:
    print(f"  [PASS] Duplicate National IDs = {dup_ids} (IDs shared across multiple sources)")
    passed += 1
else:
    print(f"  [INFO] Duplicate National IDs = {dup_ids}")
    passed += 1

print(f"\nResult: {passed}/{total} checks passed")
print("=" * 50)

---
## Quiz: Test Your Knowledge

**Q1:** Why does the CHECK_NATIONAL_ID_FORMAT regex start with `^[12]`? What would happen if a National ID started with 3?

**Q2:** The IBAN regex is `^SA[0-9A-Za-z]{22}$`. Why does it check for exactly 22 characters after 'SA' (not 24 total)?

**Q3:** Why do we store DMFs in the `DQ` schema rather than next to each table in RAW/SILVER/GOLD?

**Q4:** A custom DMF returns `NUMBER`. What does the number represent in all our DMFs?

> **What this does:** Reveals quiz answers. Try answering first!

In [ ]:
# Run this cell to reveal the answers
print("""
QUIZ ANSWERS
============

Q1: Saudi National IDs start with 1 (Saudi citizen) or 2 (resident/iqama holder).
    A value starting with 3 (like '30876543210') is INVALID -- this is one of our
    intentional test cases from the Gov Portal feed. The regex rejects it.

Q2: 'SA' is 2 characters + 22 alphanumeric = 24 total characters in a Saudi IBAN.
    The regex anchors at ^ (start) and $ (end) ensuring EXACTLY this format.
    'SA12345' fails because it has only 5 chars after 'SA' instead of 22.

Q3: Centralized DQ schema provides:
    - Single location for all quality logic (easy to audit)
    - Grantable as a unit (DQ_STEWARD role gets the whole schema)
    - No pollution of business schemas with monitoring objects
    - Easier to version and deploy as a package

Q4: The number represents the COUNT OF VIOLATIONS (records that FAIL the check).
    A return value of 0 means all records pass. This convention makes expectations
    simple: you expect VALUE = 0 for format checks.
""")

---
## Challenge (Self-Guided)

1. Create a DMF `CHECK_DQ_SCORE_THRESHOLD` that counts records where `DQ_SCORE < 70`
2. Create a DMF `CHECK_EMAIL_FORMAT` that validates email addresses with regex `^[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}$`
3. Attach your new DMFs to `INT_CUSTOMERS` and run the validation query to see results

---

**Next:** Open `3_GOLD_LAYER_DQ` to build a self-service Rules Catalog.